In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage.io
import os 
import tqdm
import glob
import tensorflow 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.data.experimental import AUTOTUNE
from tensorflow.keras import Sequential, Input, Model
from tensorflow.keras.layers import RandomRotation, RandomZoom
from tensorflow.keras.layers.experimental.preprocessing import Rescaling
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras import applications
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam


from tqdm import tqdm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from skimage.io import imread, imshow
from skimage.transform import resize

from tensorflow.keras.models import Sequential
from tensorflow.keras.metrics import Precision, AUC,Recall
from tensorflow.keras.layers import InputLayer, BatchNormalization, Dropout, Flatten, Dense, Activation, MaxPool2D, Conv2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications.densenet import DenseNet169
import copy
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
import cv2
import keras
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array
import matplotlib
import matplotlib.pylab as plt
import seaborn as sns
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix
from keras.applications.vgg16 import VGG16,preprocess_input
from keras.applications.vgg19 import VGG19,preprocess_input
from tensorflow.keras.utils import image_dataset_from_directory
import tensorflow_addons as tfa
from tensorflow_addons.optimizers import RectifiedAdam
from tensorflow_addons.optimizers import AdamW

labels = pd.read_csv('AlzheimersFLAIR/patient_labels.csv')
labels

In [ ]:
def data_augmentar():
    data_augmentation = Sequential()
    data_augmentation.add(RandomRotation(factor=(-0.15, 0.15)))
    data_augmentation.add(RandomZoom((-0.3, -0.1)))
    return data_augmentation

data_augmentation = data_augmentar()
assert(data_augmentation.layers[0].name.startswith('random_rotation'))
assert(data_augmentation.layers[0].factor == (-0.15, 0.15))
assert(data_augmentation.layers[1].name.startswith('random_zoom'))
assert(data_augmentation.layers[1].height_factor == (-0.3, -0.1))

In [ ]:
from keras.preprocessing.image import ImageDataGenerator

BATCH_SIZE=32

SEED=1345

train_datagen=ImageDataGenerator(rescale=1./255,
                                shear_range=0,
                                zoom_range=0.2, rotation_range = 0.3)

validation_datagen=ImageDataGenerator(rescale=1./255)
test_datagen=ImageDataGenerator(rescale=1./255)


#Defining directories for train,validation,test 
train_dir = 'Splitted2D/train'
validation_dir = 'Splitted2D/val'
test_dir = 'Splitted2D/test'


#Defining generatores for train,validation,test

train_generator=train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    shuffle=True,
    seed = SEED,
    batch_size= 16,
    class_mode ='categorical',
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(224, 224),
    seed = SEED,
    shuffle=True,
    batch_size= 16,
    class_mode ='categorical',
)

# Define generator for test set using flow_from_directory
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    shuffle=True,
    seed = SEED,
    batch_size = 16,
    class_mode ='categorical',
)

In [ ]:
for x, y in train_generator:
    print(type(x))

In [ ]:
for images, _ in train_generator:
    plt.figure()
    plt.imshow(images[0])
    plt.show()

In [ ]:
input_folder='jpg-slice21'

class_count=dict()
class_names=list(train_generator.class_indices.keys())
print(class_names)

for i in class_names:
    name = 'CN'
    if (i == '1'): 
        name = 'EMCI'
    elif (i == '2'):
        name = 'LMCI'
    elif (i == '3'):
        name = 'AD'
    print(name, i)
    class_count[name]=len(os.listdir(input_folder+'/'+i))

plt.figure(figsize=(10,4))
colors = ['red', 'blue', 'green', 'orange']
plt.bar(class_count.keys(),class_count.values(),color=colors)

plt.xlabel('Classes')
plt.ylabel('Counts')
plt.title('Visualization of Data Imbalance')
plt.grid(True)
plt.show()

total_samples=sum(class_count.values())

for i in range(4):
    class_weight = round(total_samples / (4 * list(class_count.values())[i]), 2)
    print(f'Weight for class \"{class_names[i]}\" : {class_weight}')

In [ ]:
filepath = './Vgg_best_weights.hdf5'
earlystopping=EarlyStopping(monitor='val_accuracy',
                           mode='max',
                           patience=15,
                           verbose=1)

checkpoint=ModelCheckpoint(filepath,monitor = 'val_accuracy', 
                                mode='max', 
                                save_best_only=True, 
                                verbose = 1)

callback_list=[earlystopping,checkpoint]

In [ ]:
input_shape = (224,224, 3)

#Create an instance of the VGG16 model
vgg16 = VGG16(include_top=False, input_shape=input_shape,
                   weights=None)

# Freeze the layers of the VGG16 model
# for layer in vgg16.layers:
#     layer.trainable = False
    
# Create a new model with additional layers
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()

prediction_layer = keras.layers.Dense(4,activation='softmax')

model = tf.keras.Sequential([vgg16, global_average_layer,
    keras.layers.BatchNormalization(),  
    keras.layers.Dropout(0.35),
    keras.layers.Dense(2048, activation=tf.keras.layers.LeakyReLU(0.1)),
    keras.layers.Dropout(0.35),
    keras.layers.Dense(512, activation=tf.keras.layers.LeakyReLU(0.1)),
    keras.layers.Dropout(0.35),
    keras.layers.Dense(256, activation=tf.keras.layers.LeakyReLU(0.1)),
    keras.layers.Dropout(0.35),
    keras.layers.Dense(64, activation=tf.keras.layers.LeakyReLU(0.1)),
    keras.layers.Dropout(0.35),
    prediction_layer
])

model.compile(optimizer=AdamW(learning_rate = 0.001, weight_decay = 0.0001), loss='categorical_crossentropy', metrics=['accuracy',tf.keras.metrics.AUC(),
                        tf.keras.metrics.Precision(),
                        tf.keras.metrics.Recall(),])

In [ ]:
model.summary()

In [ ]:
history=model.fit(train_generator,
                        validation_data=validation_generator,
                        steps_per_epoch=len(train_generator),
                        epochs = 100,
                        verbose = 1)

In [ ]:
result = model.evaluate(train_generator)
train_loss = result[0]
train_accuracy = result[1]
train_AUC = result[2]
train_pre = result[3]
train_rec = result[4]
print(f'Train Loss = {train_loss}')
print(f'Train Accuracy = {train_accuracy}')
print(f'Train AUC = {train_AUC}')
print(f'Train Precision = {train_pre}')
print(f'Train Recall = {train_rec}')

result = model.evaluate(test_generator)
test_loss = result[0]
test_accuracy = result[1]
test_AUC = result[2]
test_pre = result[3]
test_rec = result[4]
print(f'Test Loss = {test_loss}')
print(f'Test Accuracy = {test_accuracy}')
print(f'Test AUC = {test_AUC}')
print(f'Test Precision = {test_pre}')
print(f'Test Recall = {test_rec}')

In [ ]:
y_pred=[]
y_true = []
for images, label in test_generator:
    for l in label:
        y_true.append(np.argmax(l))
    
    Y_pred = [np.argmax(l) for l in model.predict(images)]
    for l in Y_pred:
        y_pred.append(l)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_confusion_matrix(cm, classes, normalize=False, cmap=plt.cm.Blues):
    if normalize:
#         cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        title = "Normalized Confusion Matrix"
    else:
        title = "Confusion Matrix"

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=90)
    plt.yticks(tick_marks, classes)

    fmt = 'd'
#     '.2f' if normalize else 
    thresh = cm.max() / 2.
    for i, j in np.ndindex(cm.shape):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

cm = confusion_matrix(y_true[:348], y_pred[:348])

class_names = [0, 1, 2, 3]

plt.figure(figsize=(8, 6))
plot_confusion_matrix(cm, class_names, normalize=True)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
# Compute the classification report
report = classification_report(y_true[:398], y_pred)
print("Classification Report:")
print(report)